## Import Python Libraries and load environment variables

In [1]:
import os
import json
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

/workspaces/hai5016-project/.venv/lib/python3.11/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

In [2]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

## Create a helper function to beautify llm responses in our Jupyter Notebook

In [3]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [4]:
# bfmsgs(result)

## Create a dummy function that returns 1500 as KRW > USD FX

In [5]:
def get_fx(currency: str) -> float:
    """Return conversion rate from KRW to the given currency (e.g., USD)."""
    # Read API key from environment variables loaded by load_dotenv()
    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing EXCHANGE_RATE_API_KEY in .env")

    # Build ExchangeRate-API endpoint with KRW as base currency
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/KRW"

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; hai5016-project/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(f"Failed to fetch exchange rates: {error}") from error

    # Optional API-level error check
    if data.get("result") != "success":
        error_type = data.get("error-type", "unknown_error")
        raise RuntimeError(f"ExchangeRate API error: {error_type}")

    rates = data.get("conversion_rates", {})
    code = currency.upper()

    if code not in rates:
        raise ValueError(f"Currency code not found: {code}")

    return float(rates[code])

In [6]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [7]:
# @tool # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from div id='wrapper'."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    # Decode HTML bytes to text
    text = raw.decode("utf-8", errors="replace")

    # Return all page text (not only div id="wrapper")
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text(separator=" ", strip=True)
    soup = BeautifulSoup(text, "html.parser")
    wrapper = soup.find("div", id="wrapper")

    if wrapper:
        return wrapper.get_text(strip=True)
    return "Wrapper div not found"

In [8]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [9]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url, get_fx],
    system_prompt=SYSTEM_PROMPT
)

- https://www.hanyang.ac.kr/re12
- http://hfspn.co/

In [10]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at university campuses? Find it at https://www.hanyang.ac.kr/re12 and http://hfspn.co/. And convert the KRW price into AZN."}]}
)
bfmsgs(result)

Here’s what I could pull from the two pages you shared, along with a KRW→AZN conversion.

Today’s menu (campus cafeterias)
- HUFSpoon / Mondayspoon (HYUP HUFS campus)
  - Monday
    - Breakfast: Dried Pollack Soup with Rice
    - Lunch 1: Kimchi Roe Rice Bowl
    - Lunch 2: Spicy Stir-Fried Pork with Rice
    - Noodles: Naengmyeon (Noodles with Cold Soup)
    - Dinner: Galbitang (Beef Rib Soup)
  - Note: This is taken from the HUFSpoon “Mondayspoon” page, which lists Monday’s daily specials.

Hanyang University re12 page
- I attempted to extract today’s menu from https://www.hanyang.ac.kr/re12, but the page text returned is highly cluttered (LOTS of navigation/information) and I could not reliably identify today’s dish names or prices from the scraped content. If you want, I can try again with a more targeted approach (e.g., a specific “Today’s Menu” section or a different campus page).

Prices and KRW-to-AZN conversion
- I did not find any price information (in KRW) on either page, so there’s nothing to convert for today’s items.
- KRW to AZN conversion rate (current): 1 KRW ≈ 0.001164 AZN
  - How to convert: AZN price = KRW price × 0.001164

Would you like me to try pulling more precise today’s menu items (and any prices) from the Hanyang re12 page, perhaps by inspecting for a specific “Today’s Menu” subsection or alternative campus pages? I can also try checking other campus menu pages if you’d like.